# Bài tập lớn 1 — SAT Encoding for Logic Puzzles
## Sudoku Solver using SAT / PySAT

**Course:** Các vấn đề hiện đại KHMT  
**Semester:** HKI 2026–2027  
**Topic:** SAT Encoding for Logic Puzzles — Sudoku Solver

This notebook is a research/implementation template designed around the assignment requirements:
- implement a Sudoku solver using SAT/PySAT;
- investigate ALO/AMO encodings such as Binary/Pairwise, Commander, Product and Sequential Counter;
- compare SAT solvers;
- compare SAT with exact approaches such as CP, CP-SAT, ILP/MIP, CPLEX and Gurobi where available;
- run experiments on instances of different difficulty/size;
- report correctness, variables, clauses/constraints, runtime and scalability;
- provide reproducible code and GitHub results.

**Important:** this notebook is a structure/template. Replace TODO sections with your actual implementation and measured results. Do not fabricate unavailable solver results.


In [33]:
from math import isqrt
from pysat.solvers import Glucose3


# 1. Research Questions

### Main question
How does the choice of SAT encoding affect Sudoku solving performance, and how does SAT compare with other exact solving approaches?

### Sub-questions
1. How can Sudoku be represented as Boolean SAT?
2. What are ALO and AMO constraints?
3. How do Pairwise, Commander, Product and Sequential Counter encodings differ?
4. How do they affect variables, clauses and solving time?
5. Which SAT solver performs best on the selected benchmark?
6. How does SAT compare with CP-SAT, CP, ILP/MIP, CPLEX and Gurobi?
7. How does performance scale with Sudoku size/difficulty?


# 2. Mathematical Formulation

For an order-$n$ Sudoku, define

$$
x_{r,c,v} = 1
$$

if cell $(r, c)$ contains value $v$.

There are

$$
n^3
$$

primary Boolean variables. For standard $9 \times 9$ Sudoku:

$$
9^3 = 729.
$$

Every cell, every row/value pair, every column/value pair, and every box/value pair must satisfy an **Exactly-One** constraint.

Exactly-One is:

$$
\operatorname{ALO}(X) \land \operatorname{AMO}(X)
$$

where:

$$
\operatorname{ALO}(x_1, \ldots, x_n) = x_1 \lor \cdots \lor x_n
$$

and AMO prohibits every pair from being simultaneously true.


# 3. Sudoku SAT Model

## Cell constraints

For every cell $(r, c)$:

$$
x_{r,c,1} \lor \cdots \lor x_{r,c,n}
$$

and an AMO constraint over those $n$ variables.

## Row constraints

For every row $r$ and value $v$:

$$
\operatorname{ExactlyOne}(x_{r,1,v}, \ldots, x_{r,n,v})
$$

## Column constraints

For every column $c$ and value $v$:

$$
\operatorname{ExactlyOne}(x_{1,c,v}, \ldots, x_{n,c,v})
$$

## Box constraints

For every box $B$ and value $v$:

$$
\operatorname{ExactlyOne}(x_{r,c,v} : (r, c) \in B)
$$

## Given clues

If the puzzle contains $S_{r,c} = v$, add the unit clause:

$$
(x_{r,c,v})
$$


# 4. ALO / AMO Encoding

This is the core experimental part.

For literals $X = \{x_1, \ldots, x_n\}$:

### ALO — At Least One

$$
(x_1 \lor x_2 \lor \cdots \lor x_n)
$$

### AMO — At Most One

No two literals may be true simultaneously.

### Exactly-One

$$
\operatorname{ALO}(X) \land \operatorname{AMO}(X)
$$

The project should compare at least:

1. Pairwise / Binary AMO
2. Commander AMO
3. Product AMO
4. Sequential Counter AMO

Keep the Sudoku model identical and change only the AMO encoding so the comparison is fair.


In [34]:
from itertools import combinations

def alo(lits):
    return [list(lits)]

def pairwise_amo(lits):
    clauses = []
    for i in range(len(lits)):
        for j in range(i + 1, len(lits)):
            clauses.append([-lits[i], -lits[j]])
    return clauses

def pairwise_exactly_one(lits):
    return alo(lits) + pairwise_amo(lits)

# TODO: implement commander_amo(lits, ...)
# TODO: implement product_amo(lits, ...)
# TODO: implement sequential_amo(lits, ...)

def sequential_amo(lits, next_var=None):
    """
    Sequential Counter Encoding for At-Most-One (AMO).

    Args:
        lits: List of SAT literals, e.g. [1, 2, 3, 4].
        next_var: ID of the first available auxiliary variable.

    Returns:
        clauses: CNF clauses enforcing AMO.
        next_var: ID of the next available variable.
    """
    n = len(lits)

    if n <= 1:
        return [], next_var

    if next_var is None:
        next_var = max(abs(lit) for lit in lits) + 1

    # s[i] indicates that at least one of the first i+1
    # literals has been selected.
    s = list(range(next_var, next_var + n - 1))
    clauses = []

    # First literal implies the first counter variable.
    clauses.append([-lits[0], s[0]])

    # Intermediate literals and counter variables.
    for i in range(1, n - 1):
        clauses.append([-lits[i], s[i]])
        clauses.append([-s[i - 1], s[i]])
        clauses.append([-lits[i], -s[i - 1]])

    # The last literal cannot be true if an earlier literal was true.
    clauses.append([-lits[n - 1], -s[n - 2]])

    return clauses, next_var + n - 1


print("hello world")


hello world


# 5. Encoding Comparison — What to Measure

For each encoding report:

- primary variables;
- auxiliary variables;
- total variables;
- total clauses;
- number of binary clauses;
- number of non-binary clauses;
- encoding time;
- solving time;
- total time;
- solved/timeout count;
- correctness.

Do not assume that fewer clauses means faster solving. SAT performance also depends on propagation structure and solver heuristics.


In [35]:
from dataclasses import dataclass

@dataclass
class EncodingStats:
    name: str
    variables: int
    auxiliary_variables: int
    clauses: int


# 6. Variable Numbering

Use a deterministic mapping from $(r, c, v)$ to a positive SAT variable ID.

With zero-based indices:

$$
\operatorname{id}(r, c, v) = ((r \cdot n) + c) \cdot n + v + 1
$$

The first $n^3$ IDs are primary Sudoku variables. Auxiliary variables should start after them.


In [36]:
def var_id(r, c, v, n):
    return (r * n + c) * n + v + 1

def decode_var(var, n):
    var -= 1
    v = var % n
    var //= n
    c = var % n
    r = var // n
    return r, c, v


# 7. Sudoku CNF Generator

Recommended architecture:

```text
SudokuCNF
 ├── add_cell_constraints()
 ├── add_row_constraints()
 ├── add_column_constraints()
 ├── add_box_constraints()
 └── add_clues()
```

Every Exactly-One constraint should call the selected ALO/AMO encoder.

The Sudoku generator must not depend on which encoding is being tested.


In [37]:
def pairwise_amo(lits):
    """Pairwise At-Most-One encoding."""
    clauses = []

    for i in range(len(lits)):
        for j in range(i + 1, len(lits)):
            clauses.append([-lits[i], -lits[j]])

    return clauses


def sequential_amo(lits, next_var):
    """Sequential Counter AMO encoding."""
    n = len(lits)

    if n <= 1:
        return [], next_var

    # Allocate n - 1 auxiliary variables.
    s = list(range(next_var, next_var + n - 1))
    clauses = []

    clauses.append([-lits[0], s[0]])

    for i in range(1, n - 1):
        clauses.append([-lits[i], s[i]])
        clauses.append([-s[i - 1], s[i]])
        clauses.append([-lits[i], -s[i - 1]])

    clauses.append([-lits[n - 1], -s[n - 2]])

    next_var += n - 1

    return clauses, next_var


def exactly_one(lits, encoding, next_var):
    """ALO + selected AMO encoding."""
    clauses = alo(lits)

    if encoding == "pairwise":
        clauses += pairwise_amo(lits)

    elif encoding == "sequential":
        amo_clauses, next_var = sequential_amo(
            lits, next_var
        )
        clauses += amo_clauses

    else:
        raise ValueError(
            "encoding must be 'pairwise' or 'sequential'"
        )

    return clauses, next_var


def sudoku_clauses(grid, encoding="pairwise"):
    """
    Convert a Sudoku grid into CNF clauses.

    Empty cells are represented by 0.
    Supports pairwise and sequential AMO encodings.

    Returns:
        clauses: CNF clauses
        max_var: largest variable ID used
    """
    n = len(grid)

    # Sudoku size must be a perfect square, e.g. 4, 9, 16.
    box_size = isqrt(n)

    if box_size * box_size != n:
        raise ValueError("Sudoku size must be a perfect square")

    if any(len(row) != n for row in grid):
        raise ValueError("grid must be an n x n matrix")

    if any(
        value < 0 or value > n
        for row in grid
        for value in row
    ):
        raise ValueError(f"Cell values must be between 0 and {n}")

    clauses = []

    # Primary variables occupy IDs 1 through n**3.
    next_var = n**3 + 1

    def add_exactly_one(lits):
        nonlocal next_var

        new_clauses, next_var = exactly_one(
            lits, encoding, next_var
        )
        clauses.extend(new_clauses)

    # 1. Cell Exactly-One:
    # Each cell contains exactly one value.
    for r in range(n):
        for c in range(n):
            lits = [
                var_id(r, c, v, n)
                for v in range(n)
            ]
            add_exactly_one(lits)

    # 2. Row/value Exactly-One:
    # Each value appears exactly once in each row.
    for r in range(n):
        for v in range(n):
            lits = [
                var_id(r, c, v, n)
                for c in range(n)
            ]
            add_exactly_one(lits)

    # 3. Column/value Exactly-One:
    # Each value appears exactly once in each column.
    for c in range(n):
        for v in range(n):
            lits = [
                var_id(r, c, v, n)
                for r in range(n)
            ]
            add_exactly_one(lits)

    # 4. Box/value Exactly-One:
    # Each value appears exactly once in each box.
    for box_r in range(0, n, box_size):
        for box_c in range(0, n, box_size):
            for v in range(n):
                lits = []

                for r in range(box_r, box_r + box_size):
                    for c in range(box_c, box_c + box_size):
                        lits.append(var_id(r, c, v, n))

                add_exactly_one(lits)

    # 5. Given clues:
    # Preserve all nonzero values in the input grid.
    for r in range(n):
        for c in range(n):
            value = grid[r][c]

            if value != 0:
                clauses.append([
                    var_id(r, c, value - 1, n)
                ])

    return clauses, next_var - 1


In [87]:
grid = [
        [8, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 3, 6, 0, 0, 0, 0, 0],
        [0, 7, 0, 0, 9, 0, 2, 0, 0],
        [0, 5, 0, 0, 0, 7, 0, 0, 0],
        [0, 0, 0, 0, 4, 5, 7, 0, 0],
        [0, 0, 0, 1, 0, 0, 0, 3, 0],
        [0, 0, 1, 0, 0, 0, 0, 6, 8],
        [0, 0, 8, 5, 0, 0, 0, 1, 0],
        [0, 9, 0, 0, 0, 0, 4, 0, 0]
    ]


# Generate CNF using Pairwise encoding.
clauses1, max_var1 = sudoku_clauses(
    grid, encoding="pairwise"
)

clauses2, max_var2 = sudoku_clauses(
    grid, encoding="sequential"
)

print("Number of clauses in pairwise:", len(clauses1))
print("Number of clauses in sequential:", len(clauses2))

print("Maximum variable ID:", max_var)
print("First 5 clauses:", clauses2[:5])



Number of clauses in pairwise: 12009
Number of clauses in sequential: 7797
Maximum variable ID: 3321
First 5 clauses: [[1, 2, 3, 4, 5, 6, 7, 8, 9], [-1, 730], [-2, 731], [-730, 731], [-2, -730]]


# 8. PySAT Solver

Recommended workflow:

```text
Sudoku instance
      ↓
CNF generator
      ↓
PySAT Solver
      ↓
SAT / UNSAT
      ↓
decode model
      ↓
independent Sudoku verification
```

Start with one baseline solver, such as Glucose, before comparing multiple SAT backends.


In [88]:
# If necessary:
# %pip install python-sat

def solve_cnf(clauses, solver_cls=Glucose3):
    solver = solver_cls()
    try:
        for clause in clauses:
            solver.add_clause(clause)
        sat = solver.solve()
        model = solver.get_model() if sat else None
        return sat, model
    finally:
        solver.delete()


In [89]:
sat, model = solve_cnf(clauses1)
if sat:
    print("the problem has an answer")
else:
    print("the probelm has no answer")

the problem has an answer


# 9. Decoding and Independent Verification

The decoder must inspect only the primary Sudoku variables. Commander/Product/Sequential encodings may introduce auxiliary variables.

Every returned model must be checked independently for:

1. every cell has exactly one value;
2. every row contains all values exactly once;
3. every column contains all values exactly once;
4. every box contains all values exactly once;
5. every original clue is preserved.

This prevents a bug in the encoding from being mistaken for a successful solver result.


In [90]:
def is_valid_sudoku(grid, original=None):
    n = len(grid)
    k = int(n ** 0.5)
    expected = set(range(1, n + 1))

    if any(set(row) != expected for row in grid):
        return False

    for c in range(n):
        if {grid[r][c] for r in range(n)} != expected:
            return False

    for br in range(0, n, k):
        for bc in range(0, n, k):
            values = {
                grid[r][c]
                for r in range(br, br+k)
                for c in range(bc, bc+k)
            }
            if values != expected:
                return False

    if original is not None:
        for r in range(n):
            for c in range(n):
                if original[r][c] != 0 and original[r][c] != grid[r][c]:
                    return False
    return True

# TODO: implement decode_sudoku(model, n)

def decode_sudoku(model, n):
    """
    Decode a SAT model into an n x n Sudoku grid.

    Args:
        model: List of signed literals returned by the SAT solver.
        n: Sudoku size, e.g. 9 for a 9x9 Sudoku.

    Returns:
        A solved Sudoku grid, or None if model is None.
    """
    if model is None:
        return None

    # Keep only positive literals: variables assigned True.
    true_vars = {lit for lit in model if lit > 0}

    grid = [[0 for _ in range(n)] for _ in range(n)]

    for r in range(n):
        for c in range(n):
            for v in range(n):
                variable = var_id(r, c, v, n)

                if variable in true_vars:
                    grid[r][c] = v + 1
                    break

            # No value assigned to this cell.
            if grid[r][c] == 0:
                raise ValueError(
                    f"No value found for cell ({r}, {c})"
                )

    return grid
    
if sat:
    solution = decode_sudoku(model, len(grid))

    for row in solution:
        print(row)

    print("Valid solution:", is_valid_sudoku(solution, grid))
else:
    print("No solution exists.")




[8, 1, 2, 7, 5, 3, 6, 4, 9]
[9, 4, 3, 6, 8, 2, 1, 7, 5]
[6, 7, 5, 4, 9, 1, 2, 8, 3]
[1, 5, 4, 2, 3, 7, 8, 9, 6]
[3, 6, 9, 8, 4, 5, 7, 2, 1]
[2, 8, 7, 1, 6, 9, 5, 3, 4]
[5, 2, 1, 9, 7, 4, 3, 6, 8]
[4, 3, 8, 5, 2, 6, 9, 1, 7]
[7, 9, 6, 3, 1, 8, 4, 5, 2]
Valid solution: True


# 10. Unit Tests

Before benchmarking:

- test a solved Sudoku;
- test a known valid Sudoku with known solution;
- test an inconsistent/UNSAT Sudoku;
- test order-4 Sudoku;
- test each encoding separately.

The same test suite must pass for every encoding.


In [91]:
def test_pairwise_encoding():
    # TODO
    pass

def test_known_sudoku():
    # TODO
    pass

def test_unsat_sudoku():
    # TODO
    pass


# 11. Benchmark Dataset

Use a fixed benchmark for every method.

Recommended categories:

| Category | Purpose |
|---|---|
| Easy | correctness/sanity |
| Medium | normal performance |
| Hard | stress test |
| Very hard | distinguish inference/encoding behavior |
| Generalized | scalability |

For every instance store:

- instance ID;
- order;
- clue count;
- difficulty;
- puzzle grid;
- known solution if available.

The paper *Sudoku as a SAT Problem* is especially useful as theoretical motivation for hard Sudoku experiments and for discussing minimal vs extended encodings.


In [92]:
from dataclasses import dataclass

@dataclass
class SudokuInstance:
    instance_id: str
    grid: list
    order: int
    clues: int
    difficulty: str = "unknown"
    known_solution: list | None = None


# 12. Benchmark Metrics

Record for every run:

### Formula
- primary variables
- auxiliary variables
- total variables
- clauses
- binary clauses
- non-binary clauses

### Runtime
- encoding/build time
- solver time
- total time

### Outcome
- SAT / UNSAT
- timeout
- error
- independently verified solution

### Solver statistics
If available:
- decisions
- conflicts
- propagations
- restarts

Use the same timeout, hardware and dataset for all comparisons.


In [60]:
import time

def benchmark_solver(clauses, solver_cls=Glucose3):
    start = time.perf_counter()
    solver = solver_cls()

    try:
        for clause in clauses:
            solver.add_clause(clause)

        build_time = time.perf_counter() - start

        t0 = time.perf_counter()
        sat = solver.solve()
        solve_time = time.perf_counter() - t0
        total_time = time.perf_counter() - start

        return {
            "status": "SAT" if sat else "UNSAT",
            "build_time": build_time,
            "solve_time": solve_time,
            "total_time": total_time,
            "model": solver.get_model() if sat else None,
        }
    finally:
        solver.delete()


# 13. Experiment A — Compare ALO/AMO Encodings

### Independent variable
Encoding:
- Pairwise
- Commander
- Product
- Sequential Counter

### Dependent variables
- variable count
- clause count
- encoding time
- solving time
- total time
- success rate

### Controlled variables
- same Sudoku instances
- same SAT solver
- same machine
- same timeout

This is the main experiment for the assignment.


In [70]:
ENCODINGS = [
    "pairwise",
    "commander",
    "product",
    "sequential",
]

encoding_results = []

# TODO:
# for instance in benchmark:
#     for encoding in ENCODINGS:
#         generate CNF
#         solve
#         verify
#         record metrics


# 14. Experiment B — Compare SAT Solvers

Potential SAT solvers depend on what is actually available in the installed PySAT environment.

Candidates may include:
- Glucose
- CaDiCaL
- Kissat
- CryptoMiniSat
- Minisat / other available backends

For a fair test, fix the encoding and dataset, then change only the solver.

**Do not report a solver as tested unless it actually ran successfully.**


In [ ]:
# TODO: inspect the installed PySAT version and available solver backends.
# Then create a dictionary of actually available solver classes.

SOLVERS = {
    "Glucose3": Glucose3,
    # "CaDiCaL": ...,
    # "Kissat": ...,
    # "CryptoMiniSat": ...,
}


# 15. Experiment C — Scalability

If reliable generalized Sudoku instances are available, test:

$$
4 \times 4, \quad 9 \times 9, \quad 16 \times 16
$$

Primary variables grow as:

$$
n^3.
$$

Examples:

$$
4^3 = 64, \quad 9^3 = 729, \quad 16^3 = 4096.
$$

Measure:
- formula size;
- encoding time;
- solving time;
- total time;
- memory/timeout behavior.

Do not assume linear runtime growth. The purpose of this section is to observe it experimentally.


# 16. Experiment D — SAT vs CP-SAT

Implement the same Sudoku model in OR-Tools CP-SAT if available.

Use the same Boolean decision-variable interpretation where practical:

$$
x_{r,c,v} \in \{0, 1\}
$$

with Exactly-One constraints.

Compare:
- SAT/PySAT
- CP-SAT

Use the same instances and timeout.


In [ ]:
# Optional:
# %pip install ortools

# TODO:
# from ortools.sat.python import cp_model
#
# def solve_with_cp_sat(grid):
#     ...


# 17. Experiment E — ILP / MIP

A 0–1 MIP formulation can use:

$$
x_{r,c,v} \in \{0, 1\}.
$$

Constraints:

$$
\sum_{v} x_{r,c,v} = 1
$$

for cells,

$$
\sum_{c} x_{r,c,v} = 1
$$

for rows,

$$
\sum_{r} x_{r,c,v} = 1
$$

for columns, and the analogous equation for each box.

If available, compare:
- Gurobi
- CPLEX MIP

The assignment also mentions CPLEX CP; implement it separately if the required environment is available.

If a commercial solver/license is unavailable, document that limitation rather than inventing results.


# 18. Unified Results Table

Use one row per method/instance/run.

Recommended columns:

```text
instance
order
clues
method
solver
encoding
variables
aux_variables
clauses
constraints
build_time
solve_time
total_time
status
valid
```

Save raw results as CSV so the report figures can be regenerated.


In [6]:
import pandas as pd

RESULT_COLUMNS = [
    "instance", "order", "clues",
    "method", "solver", "encoding",
    "variables", "aux_variables", "clauses", "constraints",
    "build_time", "solve_time", "total_time",
    "status", "valid"
]

results = pd.DataFrame(columns=RESULT_COLUMNS)
results.head()


,instance,order,clues,method,solver,encoding,variables,aux_variables,clauses,constraints,build_time,solve_time,total_time,status,valid


# 19. Data Analysis

Answer these questions using measured data:

### Encoding
- Which encoding has the smallest CNF?
- Which has the smallest number of auxiliary variables?
- Which is fastest?
- Does the best encoding change with puzzle size?

### SAT solver
- Which solver is fastest?
- Which is most consistent?
- Which handles hard instances best?

### Cross-paradigm
- SAT vs CP-SAT?
- SAT vs MIP?
- SAT vs CPLEX CP?
- SAT vs Gurobi?
- Does the ranking change for larger instances?

### Scalability
- How do variables/clauses grow?
- When does encoding time become significant?
- When does solving time dominate?


In [41]:
import matplotlib.pyplot as plt

def plot_median_runtime(df, group_col, title):
    summary = (
        df[df["status"] == "SAT"]
        .groupby(group_col)["solve_time"]
        .median()
        .sort_values()
    )
    summary.plot(kind="bar")
    plt.ylabel("Median solving time (s)")
    plt.title(title)
    plt.tight_layout()
    plt.show()

# Example:
# plot_median_runtime(results, "encoding", "Median Runtime by Encoding")


# 20. Recommended Figures for the Report

Include at least:

1. Clause count by encoding.
2. Variable count by encoding.
3. Median solving time by encoding.
4. SAT solver runtime comparison.
5. Solved percentage by method.
6. Runtime vs Sudoku difficulty.
7. Runtime vs Sudoku order.
8. Optional CDF of solving times.

Use median runtime and solved percentage in addition to mean runtime because hard-instance runtimes can contain strong outliers.


# 21. Discussion

The discussion should explain *why* results occurred.

Important theoretical point from the reference paper:

> A redundant SAT clause can be computationally useful because restricted inference such as unit propagation may exploit an explicitly present consequence much more easily than deriving that consequence through resolution.

For example, the paper's minimal and extended Sudoku encodings represent the same Sudoku solutions, but the extended encoding exposes additional constraints explicitly.

Use this idea when interpreting your own encoding experiments.

Do not assume that the encoding with fewer clauses is automatically the best.


# 22. Reproducibility / GitHub

Recommended repository:

```text
sudoku-sat/
├── notebooks/
│   └── sudoku_sat_experiments.ipynb
├── src/
│   ├── sudoku.py
│   ├── encodings.py
│   ├── solvers.py
│   ├── verification.py
│   └── benchmarks.py
├── data/
├── results/
├── figures/
├── requirements.txt
└── README.md
```

The final report should include the GitHub/other repository link as requested in the assignment.

Record:
- Python version
- OS
- CPU/RAM
- PySAT version
- solver versions
- OR-Tools version
- Gurobi/CPLEX versions if used


In [ ]:
import sys
import platform

print("Python:", sys.version)
print("OS:", platform.platform())
print("Processor:", platform.processor())

try:
    import pysat
    print("PySAT:", getattr(pysat, "__version__", "unknown"))
except ImportError:
    print("PySAT: not installed")

try:
    import pandas as pd
    print("pandas:", pd.__version__)
except ImportError:
    print("pandas: not installed")


# 23. Final Report Structure

The notebook can be converted into the final report using this structure:

## 1. Introduction
- Sudoku as a constraint problem
- Motivation for SAT
- Research questions

## 2. Background
- SAT and CNF
- ALO / AMO
- PySAT
- SAT solver concepts

## 3. Sudoku SAT Formulation
- Boolean variables
- Cell/row/column/box constraints
- Clue encoding

## 4. SAT Encodings
- Pairwise
- Commander
- Product
- Sequential Counter
- Complexity and clause/variable counts

## 5. Implementation
- architecture
- PySAT
- decoding
- verification

## 6. Experimental Methodology
- dataset
- hardware/software
- timeout
- metrics
- controlled variables

## 7. Experimental Results
- encoding comparison
- solver comparison
- scalability
- SAT vs CP/CP-SAT/MIP/CPLEX/Gurobi

## 8. Discussion
- explain observed behavior
- trade-offs
- limitations

## 9. Conclusion
- answer research questions
- summarize main findings

## References
- papers
- documentation
- datasets


# 24. Final Checklist

### Implementation
- [ ] Pairwise AMO works
- [ ] Commander works
- [ ] Product works
- [ ] Sequential Counter works
- [ ] Sudoku CNF is correct
- [ ] Model decoding is correct
- [ ] Independent verifier works
- [ ] Unit tests pass

### Experiments
- [ ] Same benchmark for all methods
- [ ] Same timeout
- [ ] Same hardware
- [ ] Encoding time recorded
- [ ] Solver time recorded
- [ ] Formula size recorded
- [ ] Results exported to CSV
- [ ] Figures generated

### Comparisons
- [ ] SAT encodings
- [ ] SAT solvers
- [ ] CP-SAT
- [ ] CP if available
- [ ] ILP/MIP
- [ ] CPLEX MIP if available
- [ ] CPLEX CP if available
- [ ] Gurobi if available

### Submission
- [ ] Report
- [ ] Notebook
- [ ] Source code
- [ ] Dataset/source
- [ ] GitHub link
- [ ] Reproducibility information
- [ ] No fabricated benchmark values
